# Bayora Blue Team - Production Classifier Training
This notebook fine-tunes a toxicity classifier using LoRA/PEFT on adversarial datasets. It is designed to run on a **Google Colab T4 GPU** runtime.

In [ ]:
# 1. Install Dependencies
# Note: If you encounter ImportErrors, please Restart Runtime (Runtime -> Restart session)
!pip install -U -q --force-reinstall transformers accelerate
!pip install -U -q datasets peft evaluate scikit-learn torchao

In [ ]:
# 2. Sanity Checks & Hardware Verification
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/bayora/blue_agent_results"
SAVE_PATH = "/content/drive/MyDrive/bayora/blue_agent_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_PATH, exist_ok=True)

In [ ]:
# 4. Load Real Datasets (ToxiGen + HH-RLHF)
from datasets import load_dataset, concatenate_datasets

def load_and_prepare_data():
    print("Loading ToxiGen (Large machine-generated set)... ")
    # skg/toxigen-data 'train' split uses 'generation' for text and 'prompt_label' for toxicity
    toxigen = load_dataset("skg/toxigen-data", name="train")["train"]
    
    # Robust mapping for ToxiGen
    def map_toxigen(x):
        content = x.get("generation") or x.get("text")
        label = x.get("prompt_label") 
        if label is None:
            label = x.get("label")
        if label is None and "toxicity_human" in x:
            label = 1 if x["toxicity_human"] >= 0.5 else 0
            
        return {"text": content, "label": label}

    toxigen = toxigen.map(map_toxigen).filter(lambda x: x["text"] is not None and x["label"] is not None).select_columns(["text", "label"])
    print(f"ToxiGen loaded: {len(toxigen)} examples")

    print("\nLoading HH-RLHF (Partial)... injection of harmful examples")
    hh = load_dataset("Anthropic/hh-rlhf", split="train[:5000]")
    hh = hh.map(lambda x: {
        "text": x["rejected"], "label": 1
    }).select_columns(["text", "label"])
    print(f"HH-RLHF loaded: {len(hh)} examples")

    full = concatenate_datasets([toxigen, hh]).shuffle(seed=42)
    return full.train_test_split(test_size=0.15, seed=42)

split = load_and_prepare_data()
train_ds = split["train"]
val_ds = split["test"]
print(f"\nCombined - Train size: {len(train_ds)} | Val size: {len(val_ds)}")

In [ ]:
# 5. Tokenization
from transformers import AutoTokenizer

MODEL_NAME = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_func(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_ds = train_ds.map(tokenize_func, batched=True)
val_ds = val_ds.map(tokenize_func, batched=True)
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
# 6. Setup Model with LoRA
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

print("Loading base model and reinitializing classifier for 2 labels...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2,
    ignore_mismatched_sizes=True  # Fixes the mismatch between 6-label checkpoint and 2-label target
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 7. Define Metrics & Training Arguments
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": f1_score(labels, preds, average="binary"),
        "accuracy": accuracy_score(labels, preds)
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none"
)

In [ ]:
# 8. Execute Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Training starting...")
trainer.train()

In [ ]:
# 9. Save Final Artifacts
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"\nSuccess! LoRA adapters and tokenizer saved to: {SAVE_PATH}")